In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

import urllib.request
import zipfile
import io


from model import *
import random
from sklearn.metrics import ConfusionMatrixDisplay, precision_score, recall_score, f1_score, adjusted_rand_score
from scipy.optimize import linear_sum_assignment
from model.initialization import init_blocks_spectral, init_spectral_with_z
from helper_functions import *

from helper_functions.Estimate_p import estimate_p_multicluster, match_clusters

In [2]:
"""
MovieLens Small Dataset — Download & Preprocessing
Produces strict per-user rankings with random tie-breaking.
"""

# ── Configuration ────────────────────────────────────────────────────────────

DATA_DIR = Path("movielens/ml-latest-small")
DOWNLOAD_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
SEED = 42          # saved for reproducibility
MIN_ITEMS = 5      # minimum strictly ranked items required to keep a user


# ── 1. Download & Extract ─────────────────────────────────────────────────────

def download_movielens(url: str = DOWNLOAD_URL, data_dir: Path = DATA_DIR) -> Path:
    """Download and extract the MovieLens small dataset if not already present."""
    if data_dir.exists():
        print(f"Data already exists at '{data_dir}', skipping download.")
        return data_dir

    print("Downloading MovieLens small dataset...")
    response = urllib.request.urlopen(url)
    zip_bytes = response.read()

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        zf.extractall("movielens")

    print(f"Extracted to '{data_dir}'.")
    return data_dir


# ── 2. Load Raw Data ──────────────────────────────────────────────────────────

def load_raw(data_dir: Path = DATA_DIR) -> dict[str, pd.DataFrame]:
    """Load all four CSV files into a dict of DataFrames."""
    files = ["ratings", "movies", "tags", "links"]
    data = {name: pd.read_csv(data_dir / f"{name}.csv") for name in files}
    print(f"Loaded: { {k: v.shape for k, v in data.items()} }")
    return data


# ── 3. Preprocess: Strict Rankings with Random Tie-Breaking ──────────────────

def make_strict_rankings(
    ratings: pd.DataFrame,
    seed: int = SEED,
    min_items: int = MIN_ITEMS,
) -> pd.DataFrame:
    """
    Convert raw ratings into strict per-user rankings.

    Ties (same rating value) are broken randomly using a fixed seed.
    Users with fewer than `min_items` strictly rankable items are dropped.

    Parameters
    ----------
    ratings   : raw ratings DataFrame (userId, movieId, rating, timestamp)
    seed      : random seed for tie-breaking — save this for reproducibility
    min_items : minimum number of items a user must have to be kept

    Returns
    -------
    DataFrame with columns: userId, movieId, rating, rank
        rank=1 means most preferred by that user.
    """
    rng = np.random.default_rng(seed)

    # Sort deterministically before adding noise so the seed maps
    # to the same rows on every run regardless of input order.
    df = (
        ratings[["userId", "movieId", "rating"]]
        .sort_values(["userId", "movieId"])
        .reset_index(drop=True)
    )

    # Add small random noise to break ties within each user's ratings.
    # Noise is scaled to be smaller than the smallest rating increment (0.5),
    # so it never changes the relative order of non-tied items.
    noise = rng.random(len(df)) * 0.1   # max noise = 0.1 << 0.5
    df["_score"] = df["rating"] + noise

    # Rank within each user: rank 1 = highest score = most preferred
    df["rank"] = (
        df.groupby("userId")["_score"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    df = df.drop(columns="_score")

    # Drop users with too few items to form a meaningful ranking
    user_counts = df.groupby("userId")["movieId"].count()
    valid_users = user_counts[user_counts >= min_items].index
    df = df[df["userId"].isin(valid_users)].reset_index(drop=True)

    n_users = df["userId"].nunique()
    n_ratings = len(df)
    avg_len = df.groupby("userId").size().mean()
    print(
        f"Strict rankings: {n_ratings} ratings across {n_users} users "
        f"(avg ranking length: {avg_len:.1f})"
    )

    return df


# ── 4. Enrich with Movie Metadata ─────────────────────────────────────────────

def enrich(rankings: pd.DataFrame, movies: pd.DataFrame) -> pd.DataFrame:
    """Join rankings with movie titles and genres."""
    return rankings.merge(movies[["movieId", "title", "genres"]], on="movieId")


# ── Main ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Step 1: download
    data_dir = download_movielens()

    # Step 2: load
    data = load_raw(data_dir)

    # Step 3: build strict rankings
    rankings = make_strict_rankings(
        data["ratings"],
        seed=SEED,
        min_items=MIN_ITEMS,
    )

    # Step 4: optionally enrich with titles
    rankings_with_titles = enrich(rankings, data["movies"])

    print(rankings_with_titles.head(20).to_string(index=False))

    # Save
    out_path = Path("movielens_strict_rankings.csv")
    rankings_with_titles.to_csv(out_path, index=False)
    print(f"\nSaved to '{out_path}'.")

Extracted to 'movielens\ml-latest-small'.
Loaded: {'ratings': (100836, 4), 'movies': (9742, 3), 'tags': (3683, 4), 'links': (9742, 3)}
Strict rankings: 100836 ratings across 610 users (avg ranking length: 165.3)
 userId  movieId  rating  rank                                     title                                      genres
      1        1     4.0   141                          Toy Story (1995) Adventure|Animation|Children|Comedy|Fantasy
      1        3     4.0   170                   Grumpier Old Men (1995)                              Comedy|Romance
      1        6     4.0   134                               Heat (1995)                       Action|Crime|Thriller
      1       47     5.0    39               Seven (a.k.a. Se7en) (1995)                            Mystery|Thriller
      1       50     5.0   116                Usual Suspects, The (1995)                      Crime|Mystery|Thriller
      1       70     3.0   202                From Dusk Till Dawn (1996)              

In [33]:

# ── MovieLens Rank Matrix (users × movies) ────────────────────────────────────
# Each row is a user, each column is a movie.
# The value is the user's rank for that movie (rank 1 = most preferred).
# Movies not rated by a user are left as NaN (partial rankings —
# analogous to the F1 partial rankings where absent drivers are missing items
# to be augmented during MCMC).
#
# MAX_MOVIES        : keep only the top-N movies by number of distinct users
#                     who rated them. Set to None to keep all movies.
# MIN_USER_RATINGS  : drop users who rated fewer than this many of the kept
#                     movies. Set to None to keep all users.

MAX_MOVIES       = 200
MIN_USER_RATINGS = 10
RANK_MATRIX_PATH = Path("movielens_rank_matrix.csv")

# Load strict rankings produced by make_strict_rankings()
rankings_df = pd.read_csv("movielens_strict_rankings.csv")

# ── Filter to top-N most-rated movies ────────────────────────────────────────
if MAX_MOVIES is not None:
    movie_user_counts = rankings_df.groupby("movieId")["userId"].nunique().sort_values(ascending=False)
    top_movies = movie_user_counts.head(MAX_MOVIES).index
    rankings_df = rankings_df[rankings_df["movieId"].isin(top_movies)].copy()
    print(f"Kept top {MAX_MOVIES} movies by user-count "
          f"(min {movie_user_counts.iloc[MAX_MOVIES - 1]} users/movie).")

    # Re-rank each user within the filtered movie subset so ranks are
    # contiguous starting from 1 (preserving relative order).
    rankings_df["rank"] = (
        rankings_df.groupby("userId")["rank"]
        .rank(method="first", ascending=True)
        .astype(int)
    )

# ── Build rank matrix ────────────────────────────────────────────────────────
rank_matrix = rankings_df.pivot(index="userId", columns="movieId", values="rank")
rank_matrix = rank_matrix.sort_index(axis=0).sort_index(axis=1)

# ── Filter users with too few rated movies ────────────────────────────────────
if MIN_USER_RATINGS is not None:
    user_rating_counts = rank_matrix.notna().sum(axis=1)
    valid_users = user_rating_counts[user_rating_counts >= MIN_USER_RATINGS].index
    n_dropped = len(rank_matrix) - len(valid_users)
    rank_matrix = rank_matrix.loc[valid_users]
    print(f"Dropped {n_dropped} users with fewer than {MIN_USER_RATINGS} rated movies "
          f"({len(valid_users)} users remain).")

n_users, n_movies = rank_matrix.shape
fill_rate = rank_matrix.notna().sum().sum() / rank_matrix.size

print(f"Rank matrix shape : {n_users} users × {n_movies} movies")
print(f"Rated entries     : {rank_matrix.notna().sum().sum():,} / {rank_matrix.size:,}  ({fill_rate:.2%} fill rate)")
print(f"Avg movies / user : {rank_matrix.notna().sum(axis=1).mean():.1f}")
print(f"Avg users / movie : {rank_matrix.notna().sum(axis=0).mean():.1f}")

rank_matrix.to_csv(RANK_MATRIX_PATH)
print(f"\nSaved to '{RANK_MATRIX_PATH}'.")


Kept top 200 movies by user-count (min 82 users/movie).
Dropped 76 users with fewer than 10 rated movies (525 users remain).
Rank matrix shape : 525 users × 200 movies
Rated entries     : 25,353 / 105,000  (24.15% fill rate)
Avg movies / user : 48.3
Avg users / movie : 126.8

Saved to 'movielens_rank_matrix.csv'.


In [32]:

# ── Convert rank matrix → partial ranking lists ───────────────────────────────
# Each user is an assessor; each movie is an item.
# Movies unrated by a user produce a shorter-than-full list (partial ranking),
# exactly as absent drivers do in the F1 pipeline.

def to_partial_rank_lists_movielens(
    rank_matrix: pd.DataFrame,
) -> tuple[list[list[int]], list[int], list[int]]:
    """
    Convert a userId×movieId rank matrix to partial ranking lists.

    Parameters
    ----------
    rank_matrix : DataFrame  (index=userId, columns=movieId, values=rank or NaN)
        Ranks are 1-based, ascending (1 = most preferred).  NaN = not rated.

    Returns
    -------
    partial_rankings : list of N lists
        Element i is a list of 0-indexed movie positions in preference order
        for that user. Length varies (partial rankings for unrated movies).
    user_ids  : list of N userIds  (row ordering)
    movie_ids : list of M movieIds (column ordering — defines item indices)
    """
    movie_ids = rank_matrix.columns.tolist()
    user_ids  = rank_matrix.index.tolist()
    movie_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

    partial_rankings: list[list[int]] = []
    for uid in user_ids:
        row = rank_matrix.loc[uid].dropna()
        # sort by rank value (ascending) → index order = preference order
        sorted_movies = row.sort_values().index.tolist()
        partial_rankings.append([movie_to_idx[mid] for mid in sorted_movies])

    return partial_rankings, user_ids, movie_ids


# ── Build the partial-ranking dataset ────────────────────────────────────────
partial_rankings, ml_user_ids, ml_movie_ids = to_partial_rank_lists_movielens(rank_matrix)
n_items_partial = len(ml_movie_ids)

lengths   = np.array([len(r) for r in partial_rankings])
miss_frac = 1.0 - lengths / n_items_partial

print(f"Total movies (n_items)  : {n_items_partial}")
print(f"Total users (assessors) : {len(partial_rankings)}")
print(f"Movies per user         : min={lengths.min()}  mean={lengths.mean():.1f}  max={lengths.max()}")
print(f"Missingness per user    : min={miss_frac.min():.1%}  mean={miss_frac.mean():.1%}  max={miss_frac.max():.1%}")
print(f"Complete rankings       : {(miss_frac == 0).sum()} / {len(partial_rankings)}")


Total movies (n_items)  : 200
Total users (assessors) : 525
Movies per user         : min=10  mean=48.3  max=194
Missingness per user    : min=3.0%  mean=75.9%  max=95.0%
Complete rankings       : 0 / 525


In [35]:
import random as _random

# ── Initialise clusters ───────────────────────────────────────────────────────
# init_spectral_with_z assumes equal-length rankings; for partial rankings we
# use a simple paired-blocks init (consecutive 0-indexed driver pairs).
# The MCMC merges/splits blocks from this neutral starting point.

C_partial = 30
gamma_partial = 1.0
delta_partial = 0.5

_blocks = [[i, i + 1] for i in range(0, n_items_partial - 1, 2)]
if n_items_partial % 2 == 1:
    _blocks.append([n_items_partial - 1])

clusters_partial = [
    ClusterParams(
        blocks=[b[:] for b in _blocks],
        theta=1.0,
        gamma=gamma_partial,
        delta=delta_partial,
    )
    for _ in range(C_partial)
]

# Round-robin z init (shuffled)
_rng_init = _random.Random(42)
z_partial_init = [i % C_partial for i in range(len(partial_rankings))]
_rng_init.shuffle(z_partial_init)

#init_mu_partial = [len(partial_rankings) / C_partial] * C_partial
init_mu_partial = [1/ C_partial] * C_partial

print(f"Initialised {C_partial} clusters over {n_items_partial} items, "
      f"{len(partial_rankings)} assessors.")
print(f"Blocks per cluster: {len(_blocks)}  "
      f"(consecutive pairs, MCMC will adjust)")


Initialised 30 clusters over 200 items, 525 assessors.
Blocks per cluster: 100  (consecutive pairs, MCMC will adjust)


In [36]:

# ── Estimate tie-penalty ──────────────────────────────────────────────────────
# estimate_p_multicluster requires equal-length rankings (it calls np.asarray).
# Strategy: use only the users whose rated-movie count equals the most common
# length as a representative sample.

_most_common_len = int(pd.Series(lengths).value_counts().idxmax())
_full_mask = [i for i, l in enumerate(lengths) if l == _most_common_len]
print(f"Estimating p_hat from {len(_full_mask)} users "
      f"with {_most_common_len} movies each "
      f"({_most_common_len / n_items_partial:.0%} coverage)")

if len(_full_mask) >= 2:
    _sub_rl = [partial_rankings[i] for i in _full_mask]
    _sub_z  = [z_partial_init[i] % C_partial for i in _full_mask]
    p_est_partial = estimate_p_multicluster(
        rankings=_sub_rl,
        cluster_assignments=_sub_z,
        delta=delta_partial,
        lam=1.0,
        n_mc_pi1=5000,
        grid_size=1000,
    )
    p_hat_partial = p_est_partial["p_hat_global"]
    print(f"Estimated p_hat: {p_hat_partial:.4f}")
else:
    p_hat_partial = 0.5   # sensible default when no common-length subset exists
    print(f"Using default p_hat: {p_hat_partial:.4f}")


Estimating p_hat from 15 users with 28 movies each (14% coverage)


c:\Users\jonanord\OneDrive - Universitetet i Oslo\Documents\Code\TiedBayesMallows\helper_functions\Estimate_p.py:565: UserWarning: Cluster 2 has < 2 assessors; skipping.
  warnings.warn(f"Cluster {c} has < 2 assessors; skipping.")
c:\Users\jonanord\OneDrive - Universitetet i Oslo\Documents\Code\TiedBayesMallows\helper_functions\Estimate_p.py:565: UserWarning: Cluster 4 has < 2 assessors; skipping.
  warnings.warn(f"Cluster {c} has < 2 assessors; skipping.")
c:\Users\jonanord\OneDrive - Universitetet i Oslo\Documents\Code\TiedBayesMallows\helper_functions\Estimate_p.py:565: UserWarning: Cluster 8 has < 2 assessors; skipping.
  warnings.warn(f"Cluster {c} has < 2 assessors; skipping.")
c:\Users\jonanord\OneDrive - Universitetet i Oslo\Documents\Code\TiedBayesMallows\helper_functions\Estimate_p.py:565: UserWarning: Cluster 13 has < 2 assessors; skipping.
  warnings.warn(f"Cluster {c} has < 2 assessors; skipping.")
c:\Users\jonanord\OneDrive - Universitetet i Oslo\Documents\Code\TiedBayesM

Estimated p_hat: 0.3338


c:\Users\jonanord\OneDrive - Universitetet i Oslo\Documents\Code\TiedBayesMallows\helper_functions\Estimate_p.py:565: UserWarning: Cluster 29 has < 2 assessors; skipping.
  warnings.warn(f"Cluster {c} has < 2 assessors; skipping.")


In [37]:
p_hat_partial

0.3338334984984985

In [38]:

# ── MCMC ──────────────────────────────────────────────────────────────────────
RUNS_DIR = Path("movielens/runs")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

#p_hat_partial = 0.4

model_partial = MixtureRankingModel(
    rankings=partial_rankings,
    n_items=n_items_partial,      # full movie roster (unrated movies are missing items)
    init_clusters=clusters_partial,
    init_z=z_partial_init,
    init_mu=init_mu_partial,
    seed=42,
    verbose=True,
    init_theta=1.0,
)

final_state_partial, samples_partial = model_partial.run_mcmc(
    n_iter=15000,
    burn_in=10000,
    thin=1,
    save_samples=True,
    save_tau=True,
    save_theta=True,
    n_item_moves_per_cluster=1,
    gamma=gamma_partial,
    delta=delta_partial,
    a_theta=4,
    b_theta=1,
    theta_jump=10,
    ranking_jump=5,              # augment missing movies every 5th iteration
    use_annealing=True,
    temp_min=0.25,
    temp_max=1.0,
    tie_penalty=p_hat_partial,
)

model_partial.print_acceptance_summary()

if samples_partial is not None:
    summaries_partial = model_partial.estimate_map(samples_partial, ci=0.95)
    print("MAP summaries:", summaries_partial)


[Model] Initialized: N=525 assessors, n=200 items, C=30 clusters, n_pairs=19,900
[Model] Compute:  CPU  (GPU unavailable — torch not installed)
[Model] U_all:    525×19,900  (39.9 MB float32, on CPU)
[Model] Parallel: N threshold = disabled
[Model] Partial rankings: 525/525 assessors have missing items
[Model]   Missing items per assessor: min=6, max=190, mean=151.7
  Cluster 0: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 1: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 2: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 3: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 4: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 5: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 6: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 7: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 8: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 9: 100 blocks, theta=1.000, gamma=1.000, delta=0.500
 

In [ ]:

# ── Collapsed-cluster diagnostics ─────────────────────────────────────────────
# Tracks which clusters have 0 assessors at each saved sample and tests whether
# any cluster that went empty later recovered (even transiently).
#
# Model behaviour recap (from core.py):
#   _update_cluster_blocks  → returns immediately when Rc is empty ✓
#   _update_cluster_theta   → returns immediately when n_c == 0    ✓
#   _update_tau             → still draws tau[c] ~ Dir(prior_mu[c])
#                             so tau[c] > 0 always → recovery IS possible
#   _compute_all_disagreements → still computes all C columns (wasted work)
#   _update_z               → still evaluates log-weight for all C clusters

z_arr = np.array(samples_partial.z_samples, dtype=np.intp)   # (T, N)
T, N  = z_arr.shape

# Cluster sizes at each sample: (T, C)
occ = np.stack([np.bincount(z_arr[t], minlength=C_partial) for t in range(T)])

empty_mask = (occ == 0)      # True where cluster was empty
ever_empty = empty_mask.any(axis=0)
ever_nonempty_after_empty = np.zeros(C_partial, dtype=bool)

for c in range(C_partial):
    if not ever_empty[c]:
        continue
    first_empty = np.argmax(empty_mask[:, c])      # first time the cluster went to 0
    recovered_after = occ[first_empty:, c] > 0     # any non-zero size after that?
    ever_nonempty_after_empty[c] = recovered_after.any()

print("── Cluster collapse summary ──────────────────────────────────")
print(f"  Total samples (post-burn-in): {T}")
print(f"  Clusters total              : {C_partial}")
print(f"  Clusters never empty        : {(~ever_empty).sum()}")
print(f"  Clusters ever empty         : {ever_empty.sum()}")
print(f"  Clusters empty the whole run: {empty_mask.all(axis=0).sum()}")
print()

if ever_empty.any():
    print(f"  Of the {ever_empty.sum()} clusters that went empty:")
    recoveries = ever_nonempty_after_empty[ever_empty].sum()
    print(f"    Recovered at least once : {recoveries}")
    print(f"    Stayed empty permanently: {ever_empty.sum() - recoveries}")
    print()

    print(f"  {'Cluster':>8}  {'Min size':>8}  {'Max size':>8}  {'% empty':>8}  {'Recovered?':>12}")
    for c in np.where(ever_empty)[0]:
        pct = empty_mask[:, c].mean() * 100
        print(f"  {c:>8}  {occ[:, c].min():>8}  {occ[:, c].max():>8}  {pct:>7.1f}%  {'yes' if ever_nonempty_after_empty[c] else 'no — permanently collapsed':>12}")

# ── Visual: cluster occupancy heatmap over samples ────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(occ.T, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_xlabel("Sample index (post-burn-in)")
ax.set_ylabel("Cluster")
ax.set_title("Cluster occupancy over MCMC samples  (darker = more assessors)")
plt.colorbar(im, ax=ax, label="# assessors")
plt.tight_layout()
plt.show()


In [ ]:

import json
import pickle
from datetime import datetime

# ── Find MAP ──────────────────────────────────────────────────────────────────
map_result_partial = model_partial.find_map(samples_partial, refine=True, verbose=True)

for c, cl in enumerate(map_result_partial["clusters"]):
    print(f"Cluster {c}: {cl['blocks']}")

# ── Save run ──────────────────────────────────────────────────────────────────
run_name = f"ml_partial_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_dir  = RUNS_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)

# 1. MCMC parameters — human-readable JSON
mcmc_params_ml = dict(
    n_assessors              = len(partial_rankings),
    n_items                  = n_items_partial,
    max_movies               = MAX_MOVIES,
    C                        = C_partial,
    gamma                    = gamma_partial,
    delta                    = delta_partial,
    p_hat                    = p_hat_partial,
    n_iter                   = 15000,
    burn_in                  = 10000,
    thin                     = 2,
    n_item_moves_per_cluster = 1,
    a_theta                  = 4,
    b_theta                  = 1,
    theta_jump               = 10,
    ranking_jump             = 5,
    use_annealing            = True,
    temp_min                 = 0.25,
    temp_max                 = 1.0,
    run_name                 = run_name,
    saved_at                 = datetime.now().isoformat(),
)
with open(run_dir / "mcmc_params.json", "w") as f:
    json.dump(mcmc_params_ml, f, indent=2)

# 2. MAP result — JSON
map_serialisable = {
    "best_t":       int(map_result_partial["best_t"]),
    "logp_chain":   float(map_result_partial["logp_chain"]),
    "logp_refined": float(map_result_partial.get("logp_refined", map_result_partial["logp_chain"])),
    "z":            [int(x) for x in map_result_partial["z"]],
    "clusters": [
        {"blocks": cl["blocks"], "theta": float(cl["theta"])}
        for cl in map_result_partial["clusters"]
    ],
}
with open(run_dir / "map_result.json", "w") as f:
    json.dump(map_serialisable, f, indent=2)

# 3. User and movie metadata
pd.DataFrame({"userId":  ml_user_ids}).to_csv(run_dir / "users.csv",  index=False)
pd.DataFrame({"movieId": ml_movie_ids}).to_csv(run_dir / "movies.csv", index=False)

# 4. Posterior samples and final state (binary)
with open(run_dir / "samples.pkl", "wb") as f:
    pickle.dump(samples_partial, f)
with open(run_dir / "final_state.pkl", "wb") as f:
    pickle.dump(final_state_partial, f)

print(f"\nSaved run to: {run_dir}")


In [34]:
print("=== partial_rankings ===")
print(f"Type          : {type(partial_rankings)}")
print(f"Length        : {len(partial_rankings)}  (one list per user/assessor)")
print()

for i in range(3):
    uid = ml_user_ids[i]
    r   = partial_rankings[i]
    movies_in_order = [ml_movie_ids[idx] for idx in r]
    print(f"User {uid} (ranked {len(r)}/{n_items_partial} movies):")
    print(f"  Item indices (0-based) : {r[:8]}{'...' if len(r) > 8 else ''}")
    print(f"  movieIds (pref order)  : {movies_in_order[:8]}{'...' if len(r) > 8 else ''}")
    print()

print("=== ml_movie_ids ===")
print(f"Length  : {len(ml_movie_ids)}   (item index → movieId lookup)")
print(f"First 5 : {ml_movie_ids[:5]}")
print()
print("=== ml_user_ids ===")
print(f"Length  : {len(ml_user_ids)}   (assessor index → userId lookup)")
print(f"First 5 : {ml_user_ids[:5]}")


=== partial_rankings ===
Type          : <class 'list'>
Length        : 525  (one list per user/assessor)

User 1 (ranked 70/200 movies):
  Item indices (0-based) : [113, 139, 73, 134, 24, 143, 10, 76]...
  movieIds (pref order)  : [1617, 2716, 919, 2571, 231, 2858, 47, 1073]...

User 4 (ranked 50/200 movies):
  Item indices (0-based) : [83, 59, 128, 62, 73, 141, 77, 72]...
  movieIds (pref order)  : [1196, 593, 2174, 608, 919, 2791, 1080, 912]...

User 5 (ranked 28/200 movies):
  Item indices (0-based) : [57, 49, 30, 60, 6, 14, 40, 11]...
  movieIds (pref order)  : [590, 527, 296, 595, 21, 110, 367, 50]...

=== ml_movie_ids ===
Length  : 200   (item index → movieId lookup)
First 5 : [1, 2, 6, 10, 16]

=== ml_user_ids ===
Length  : 525   (assessor index → userId lookup)
First 5 : [1, 4, 5, 6, 7]
